In [ ]:
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tqdm

from ase import units
from ase.atoms import Atoms
from ase.build import molecule
from torch_dftd.torch_dftd3_calculator import TorchDFTD3Calculator
from ase.calculators.dftd3 import DFTD3

from cc2cc.utils import gen_mole


class Model(nn.Module):
    """
    Fully connected neural network (dense network)
    """

    def __init__(self, device="cuda", damping="zero", **kwargs):
        super().__init__()

        # device="cuda:0" for fast GPU computation.
        self.calc = TorchDFTD3Calculator(
            device=device,
            dtype=torch.float64,
            xc="b3-lyp",
            damping=damping,
            bidirectional=False,
        )

        if damping == "zero":
            self.param_vector = torch.nn.Parameter(
                torch.tensor(
                    [
                        kwargs.get("rs6", 1.261),
                        kwargs.get("s18", 1.703),
                    ],
                    dtype=torch.float64,
                    device=device,
                )
            )
            self.params = {
                "s6": kwargs.get("s6", 1.0),
                "rs6": self.param_vector[0],
                "s18": self.param_vector[1],
                "rs18": kwargs.get("rs18", 1.0),
                "alp": kwargs.get("alp", 14.0),
            }
        elif damping == "bj":
            self.param_vector = torch.nn.Parameter(
                torch.tensor(
                    [
                        kwargs.get("rs6", 0.3981),
                        kwargs.get("s18", 1.9889),
                        kwargs.get("rs18", 4.4211),
                    ],
                    dtype=torch.float64,
                    device=device,
                )
            )
            self.params = {
                "s6": kwargs.get("s6", 1.0),
                "rs6": self.param_vector[0],
                "s18": self.param_vector[1],
                "rs18": self.param_vector[2],
                "alp": kwargs.get("alp", 14.0),
            }
        self.calc.dftd_module.params = self.params
        self.damping = damping

    def forward(self, batch_dicts):
        self.calc.reset()

        # Calculate the energy using the DFTD3 calculator
        E_disp = self.calc.dftd_module.calc_energy_batch(
            **batch_dicts, damping=self.damping
        )

        return E_disp * units.mol / units.kcal

    def obtain_batch_dicts(self, atoms_list):
        # Calculator.calculate(self, atoms, properties, system_changes)
        input_dicts_list = [self.calc._preprocess_atoms(atoms) for atoms in atoms_list]
        # --- Make batch ---
        n_nodes_list = [d["Z"].shape[0] for d in input_dicts_list]
        shift_index_array = torch.cumsum(torch.tensor([0] + n_nodes_list), dim=0)
        cell_batch = torch.stack(
            [
                (
                    torch.eye(3, device=self.calc.device, dtype=self.calc.dtype)
                    if d["cell"] is None
                    else d["cell"]
                )
                for d in input_dicts_list
            ]
        )

        batch_dicts = dict(
            Z=torch.cat([d["Z"] for d in input_dicts_list], dim=0),  # (n_nodes,)
            pos=torch.cat([d["pos"] for d in input_dicts_list], dim=0),  # (n_nodes,)
            cell=cell_batch,  # (bs, 3, 3)
            pbc=torch.stack([d["pbc"] for d in input_dicts_list]),  # (bs, 3)
            shift_pos=torch.cat(
                [d["shift_pos"] for d in input_dicts_list], dim=0
            ),  # (n_nodes,)
        )
        batch_dicts["edge_index"] = torch.cat(
            [
                d["edge_index"] + shift_index_array[i]
                for i, d in enumerate(input_dicts_list)
            ],
            dim=1,
        )
        batch_dicts["batch"] = torch.cat(
            [
                torch.full((n_nodes,), i, dtype=torch.long, device=self.calc.device)
                for i, n_nodes in enumerate(n_nodes_list)
            ],
            dim=0,
        )
        batch_dicts["batch_edge"] = torch.cat(
            [
                torch.full(
                    (d["edge_index"].shape[1],),
                    i,
                    dtype=torch.long,
                    device=self.calc.device,
                )
                for i, d in enumerate(input_dicts_list)
            ],
            dim=0,
        )

        batch_dicts["pos"].requires_grad_(True)
        return batch_dicts


data = pd.read_csv(
    "/home/dhem/workspace/2025.1/validate/ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ.csv"
)
data_name_list = (data["name"].str.split("_cc-pVDZ").str[0]).to_numpy()
data_cc_ene = data["cc_ene"].to_numpy() * 627.5094733748099
data_dft_ene = data["scf_ene"].to_numpy() * 627.5094733748099
batch_subset = [
    "W4_11",
    "G21EA",
    "G21IP",
    "DIPCS10",
    "PA26",
    "SIE4x4",
    "ALKBDE10",
    "YBDE18",
    "AL2X6",
    "HEAVYSB11",
    "NBPRC",
    "ALK8",
    "RC21",
    "G2RC",
    "BH76RC",
    "FH51",
    "TAUT15",
    "DC13",
    "MB16_43",
    "DARC",
    "RSE43",
    "BSR36",
    "CDIE20",
    "ISO34",
    # "ISOL24",
    # "C60ISO",
    "PArel",
    "BH76",
    "BHPERI",
    "BHDIV10",
    "INV24",
    "BHROT27",
    "PX13",
    "WCPT18",
    "RG18",
    "ADIM6",
    "S22",
    "S66",
    # "HEAVY28",
    "WATER27",
    "CARBHB12",
    "PNICO23",
    "HAL59",
    "AHB21",
    "CHB6",
    "IL16",
    "IDISP",
    "ICONF",
    "ACONF",
    "Amino20x4",
    "PCONF21",
    "MCONF",
    "SCONF",
    # "UPU23",
    "BUT14DIOL",
]

with open(f"new_dataset/gmtkn-cc-pVDZ.json") as f:
    json_data = json.load(f)

input_batch = {}
name_batch_list = {}
weight_batch_list = {}
mean_absolute_deviation = []
# model = Model(device="cuda", damping="bj")
model = Model(device="cuda", damping="zero")
model.compile(mode="max-autotune-no-cudagraphs")
for name_mol in data_name_list:
    for i_subset in batch_subset:
        if i_subset == "BH76RC":
            i_subset_name = "BH76"
        else:
            i_subset_name = i_subset
        if name_mol.startswith(i_subset_name):
            mol = gen_mole(name_mol, 0, 1, 0, "cc-pVDZ", True, "gmtkn-cc-pVDZ")
            atoms = Atoms(
                symbols=mol.elements, positions=mol.atom_coords() * units.Bohr
            )
            if i_subset not in input_batch:
                input_batch[i_subset] = []
            input_batch[i_subset].append(atoms)
            if i_subset not in name_batch_list:
                name_batch_list[i_subset] = []
            name_batch_list[i_subset].append(name_mol)

for i_subset in batch_subset:
    if i_subset == "BH76RC":
        i_subset_name = "BH76"
    else:
        i_subset_name = i_subset
    reaction_dict = json_data[f"reaction-{i_subset}"]
    name_batch_list[i_subset] = np.array(name_batch_list[i_subset])
    input_batch[i_subset] = model.obtain_batch_dicts(input_batch[i_subset])
    reaction_dict_copy = reaction_dict.copy()
    for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
        reaction_dict_copy.items()
    ):
        systems_list = i_reaction["systems"]
        stoichiometry_list = i_reaction["stoichiometry"]

        for i in range(len(systems_list)):
            if i_subset == "BH76RC":
                mole_name = f"{systems_list[i]}"
            else:
                mole_name = f"{i_subset}-{systems_list[i]}"
            stoichiometry = int(stoichiometry_list[i])

            if mole_name in json_data:
                if isinstance(json_data[mole_name], str):
                    mole_name = json_data[mole_name]

            col = np.where(data_name_list == mole_name)[0]
            if col.size != 1:
                print(f"Warning: {mole_name} not found in name_list")
                reaction_dict.pop(i_reaction_keys)
                break
    json_data[f"reaction-{i_subset}"] = reaction_dict

energy_batch_target = {}
for i_subset in batch_subset:
    if i_subset == "BH76RC":
        i_subset_name = "BH76"
    else:
        i_subset_name = i_subset
    reaction_dict = json_data[f"reaction-{i_subset}"]

    energy_batch_target[i_subset] = torch.zeros(
        len(reaction_dict), dtype=torch.float64, device="cpu"
    )
    weight_batch = np.zeros(len(reaction_dict), dtype=np.float64)
    for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
        reaction_dict.items()
    ):
        systems_list = i_reaction["systems"]
        stoichiometry_list = i_reaction["stoichiometry"]
        energy_dft = 0

        for i in range(len(systems_list)):
            if i_subset == "BH76RC":
                mole_name = f"{systems_list[i]}"
            else:
                mole_name = f"{i_subset}-{systems_list[i]}"
            stoichiometry = int(stoichiometry_list[i])

            if mole_name in json_data:
                if isinstance(json_data[mole_name], str):
                    mole_name = json_data[mole_name]

            col = np.where(data_name_list == mole_name)[0]
            energy_dft += (data_cc_ene[col[0]] - data_dft_ene[col[0]]) * stoichiometry
            weight_batch[i_reaction_name] += data_cc_ene[col[0]] * stoichiometry
        energy_batch_target[i_subset][i_reaction_name] = energy_dft
    mean_absolute_deviation.extend(np.abs(weight_batch))
    weight_batch_list[i_subset] = 1 / np.mean(np.abs(weight_batch))

print(
    f"mean_absolute_deviation: {np.mean(mean_absolute_deviation) / len(mean_absolute_deviation)}"
)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-5)
loss_function = torch.nn.L1Loss(reduction="sum")
torch.set_printoptions(precision=10)
energy_batch_output = {}
print("start training...")

for epoch in tqdm.tqdm(range(2501)):
    loss_batch = []
    wtmad_2 = 0
    optimizer.zero_grad()
    for i_subset in batch_subset:
        energy = model(input_batch[i_subset])

        reaction_dict = json_data[f"reaction-{i_subset}"]
        energy_batch_output[i_subset] = torch.zeros(
            len(reaction_dict), dtype=torch.float64
        )
        for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
            reaction_dict.items()
        ):
            systems_list = i_reaction["systems"]
            stoichiometry_list = i_reaction["stoichiometry"]
            energy_dft = 0

            for i in range(len(systems_list)):
                mole_name = (
                    systems_list[i]
                    if i_subset == "BH76RC"
                    else f"{i_subset}-{systems_list[i]}"
                )
                stoichiometry = int(stoichiometry_list[i])

                if mole_name in json_data:
                    if isinstance(json_data[mole_name], str):
                        mole_name = json_data[mole_name]

                col_disp = np.where(name_batch_list[i_subset] == mole_name)[0]
                if col_disp.size == 1:
                    energy_dft += energy[col_disp[0]] * stoichiometry
                else:
                    print(f"Warning: {mole_name} not found in name_list")
                    break
            energy_batch_output[i_subset][i_reaction_name] = energy_dft
        loss = (
            loss_function(energy_batch_output[i_subset], energy_batch_target[i_subset])
            * weight_batch_list[i_subset]
        )
        loss_batch.append(
            torch.mean(
                torch.abs(energy_batch_output[i_subset] - energy_batch_target[i_subset])
            ).item()
        )
        wtmad_2 += (
            torch.sum(
                torch.abs(energy_batch_output[i_subset] - energy_batch_target[i_subset])
            )
            * weight_batch_list[i_subset]
        ).item()
        # clip the loss to avoid exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch: {epoch}, wtmad_2: {wtmad_2 * np.mean(mean_absolute_deviation) / len(mean_absolute_deviation)}, loss: {loss_batch}"
        )

print(f"params_vector {model.params}")

mean_absolute_deviation: 0.051378935483546606
start training...


  0%|          | 1/2501 [01:16<53:23:00, 76.87s/it]

Epoch: 0, wtmad_2: 18.51404273011612, loss: [1.2488292262185574, 0.8236741569613609, 0.6851971759009534, 2.049049567035767, 2.9518911540569857, 3.7298475837035125, 0.949787882551871, 4.254999656373982, 2.633388909623902, 1.845095711826513, 2.1921333994276764, 2.009979872016414, 1.8167919893766131, 2.145181980729672, 0.9649400847075775, 2.136611971160006, 1.5787714070837617, 5.746533531215463, 21.114756888126728, 3.0602324895426056, 2.7550105112203784, 2.3688558323909334, 1.4796521028906167, 1.7085878798990228, 2.907650502974788, 3.0935178880214504, 5.4917782995311475, 4.850282134275199, 3.631610015610844, 1.7629904263046436, 1.2128061044147282, 2.7496629400742005, 0.8196585185859537, 2.778132768722808, 3.1967043272628577, 1.834271746992449, 9.176914442307949, 1.029996354792146, 1.4465485269122857, 2.5625522443903614, 2.212733377582248, 1.4329623781426046, 4.059323761554237, 6.653103113066623, 0.9053750749257725, 2.444587245151729, 1.2406241203059092, 2.4522413597387844, 1.8547458595689

  4%|▍         | 101/2501 [03:11<44:23,  1.11s/it] 

Epoch: 100, wtmad_2: 18.341157776964717, loss: [1.2453792042806273, 0.8235786372212414, 0.6852616782137547, 2.048737054805976, 2.950814791783873, 3.7230557832498694, 0.9498546539287138, 4.2726132335718585, 2.6471135039922804, 1.8413300693532133, 2.204441955300046, 2.0286121707480724, 1.8076040767421202, 2.1389605255926174, 0.9635636343076749, 2.1319414043436193, 1.5769167152204053, 5.7517904140596, 20.924461387286684, 3.115861917564968, 2.760183162010692, 2.362059460602755, 1.4794948298944013, 1.712768321104202, 2.909569398458997, 3.0885807718260825, 5.4438872105028455, 4.850264848765854, 3.627951187491718, 1.7620521913070784, 1.2078915973324427, 2.7402061835183944, 0.8046675888305251, 2.696580491129428, 3.1412392916096494, 1.7803857894816313, 9.02952998917418, 1.0131070241761633, 1.4291398338505392, 2.538467938759796, 2.2046124619426837, 1.4344091232571778, 4.011646288767832, 6.724612853418499, 0.9052203895253336, 2.4529095029962718, 1.2387209038224885, 2.3932209180876103, 1.815912431

...

 48%|████▊     | 1201/2501 [24:02<24:53,  1.15s/it]

Epoch: 1200, wtmad_2: 16.773193144250293, loss: [1.214460863943092, 0.8227598199639469, 0.6858892642378636, 2.0470809304021644, 2.9452907588923853, 3.658895860191195, 0.9505279330290044, 4.434082380246033, 3.005808410757883, 1.8043421945238844, 2.3206146190312853, 2.2293205186882474, 1.7502627065760972, 2.084606782432963, 0.9509134610085184, 2.1068722695910522, 1.5869389532320228, 5.780738496299279, 19.231924710278417, 3.714258625791262, 2.804734814061901, 2.3972551749318125, 1.4801492252527364, 1.7492949390892012, 2.925301489871939, 3.0436321778082553, 5.001062086866794, 4.843078269341026, 3.5929841957932758, 1.7550439918557244, 1.1749872647895443, 2.652359572626741, 0.6636462229915367, 1.9267627715625704, 2.60728309021452, 1.269803710841469, 7.525608780476631, 0.8553911834996174, 1.2658807965431746, 2.3074770988663658, 2.131943595099377, 1.4450578877819997, 3.567547078611967, 7.4676937956876115, 0.904384290495931, 2.5380181788859053, 1.2227124665272815, 1.900133476631935, 1.468986474

 52%|█████▏    | 1301/2501 [25:55<23:13,  1.16s/it]

Epoch: 1300, wtmad_2: 16.664341031743948, loss: [1.2122234936659448, 0.8227011304108988, 0.6859424487954884, 2.0470341614125678, 2.9452315710280916, 3.6541222873376, 0.950583697723344, 4.446200316459972, 3.0409903343010996, 1.8013748410615205, 2.329205548882254, 2.245914290086591, 1.7484537575856411, 2.0815679489760037, 0.9499868489330242, 2.107328927444753, 1.5875870149420748, 5.782342023839633, 19.105666257299887, 3.759027122716485, 2.8078435890204134, 2.404142336899432, 1.4803472638962774, 1.7518842231697604, 2.926233988997627, 3.040478132705348, 4.969227843124192, 4.842243297881499, 3.590166592772474, 1.75467383809649, 1.1733622446275602, 2.6458670278982144, 0.6537996601314259, 1.8706093116157578, 2.5672361551844203, 1.2332168772448078, 7.407283589142126, 0.8433867021134107, 1.2536443696510982, 2.28981160447678, 2.1268181415721377, 1.445884839199925, 3.534664864444935, 7.529953404758199, 0.9042968617645505, 2.5448911516599813, 1.2218095699948104, 1.8659413723469054, 1.4473755889707

 56%|█████▌    | 1389/2501 [27:34<21:10,  1.14s/it]

In [ ]:
data_dft_bj = []

for name_mol in data_name_list:
    mol = gen_mole(name_mol, 0, 1, 0, "cc-pVDZ", True, "gmtkn-cc-pVDZ")
    atoms = Atoms(
        symbols=mol.elements, positions=mol.atom_coords() * units.Bohr
    )
    energy = model(model.obtain_batch_dicts([atoms]))
    print(f"{name_mol}: {energy.item():.10f} kcal/mol")
    data_dft_bj.append(energy.item() / 627.5094733748099)

data["modified_ai_d3zero"] = data_dft_bj
data.to_csv(
    "/home/dhem/workspace/2025.1/validate/ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ.csv"
)

W4_11-al: 0.0000000000 kcal/mol
W4_11-b: 0.0000000000 kcal/mol
W4_11-be: 0.0000000000 kcal/mol
W4_11-c: 0.0000000000 kcal/mol
W4_11-cl: 0.0000000000 kcal/mol
W4_11-f: 0.0000000000 kcal/mol
W4_11-h: 0.0000000000 kcal/mol
W4_11-n: 0.0000000000 kcal/mol
W4_11-o: 0.0000000000 kcal/mol
W4_11-p: 0.0000000000 kcal/mol
W4_11-s: 0.0000000000 kcal/mol
W4_11-si: 0.0000000000 kcal/mol
W4_11-alcl: -0.3576075242 kcal/mol
W4_11-alf: -0.1368629922 kcal/mol
W4_11-alh: -0.1372683986 kcal/mol
W4_11-b2: -0.2599370008 kcal/mol
W4_11-be2: -0.5624994504 kcal/mol
W4_11-bf: -0.1019554532 kcal/mol
W4_11-bh: -0.0929963745 kcal/mol
W4_11-bn: -0.1828000318 kcal/mol
W4_11-bn3pi: -0.1827815905 kcal/mol
W4_11-c2: -0.2067492650 kcal/mol
W4_11-cf: -0.0962838703 kcal/mol
W4_11-ch: -0.0818857188 kcal/mol
W4_11-cl2: -0.2959478108 kcal/mol
W4_11-clf: -0.1230110298 kcal/mol
W4_11-clo: -0.1595156176 kcal/mol
W4_11-cn: -0.1664631665 kcal/mol
W4_11-co: -0.1278155284 kcal/mol
W4_11-cs: -0.2753973463 kcal/mol
W4_11-f2: -0.054920